In [ ]:
from IPython.display import display, HTML
import pandas as pd
from pathlib import Path
import numpy as np
import os

import japanize_matplotlib
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams['figure.dpi'] = 100
import seaborn as sns
sns.set(context='notebook', style='whitegrid', font='IPAexGothic')

result_dir = Path(os.environ['INPUT_DIR'])
out_dir = result_dir / 'flex'
out_image_dir = out_dir / 'image'
out_image_dir.mkdir(parents=True, exist_ok=True)
image_dir = Path('../../data/annotation/images/ap_whole')
config_filename = '../../data/annotation/config.yaml'
ref_subdir = 'reference'
auto_subdir = 'auto'

df_flex = pd.read_csv(result_dir / 'ap_flex_completed_set.csv')
df_flex['file_id'] = df_flex['filepath'].map(lambda e: '_'.join(Path(e).parts[1:]).replace('.jpg',''))

In [ ]:
from szkspyne import settings, measure_and_draw
import szkspyne.utils.measure as measure_utils
import json
import itertools


styles = '''
<style>
div {
    display: flex;
    flex-direction: row;
}
svg {
    max-height : 1024px;
}
</style>
'''

HEIGHT = 512
POS = ['neut','left','right']
LABEL_NAMES = settings.VERTEBRA_LABEL_NAMES

all_data_dict, all_lm_data_dict = {}, {}
for subdir in [ref_subdir, auto_subdir]:
    ndjson_fn = result_dir / subdir / 'measures_ap_whole.ndjson'
    all_data_dict[subdir], all_lm_data_dict[subdir] = {}, {}
    if ndjson_fn.exists():
        with open(ndjson_fn) as f:
            all_data = [json.loads(line) for line in f]
        for d in all_data:
            all_data_dict[subdir][d['filename']] = d
        with open(result_dir / subdir / 'keypoints/ap_whole/all_final_keypoints.ndjson') as f:
            all_lm_data = [json.loads(line) for line in f]
        for d in all_lm_data:
            all_lm_data_dict[subdir][d['filename']] = d


rows = []
pid_date_pairs = []
for (pid, date), df_study in df_flex.groupby(['PatientID', 'StudyDate'], sort=False):
    pid_date_pairs.append((pid, date))
    angles_per_subdir = []
    svg_names = []
    for subdir in [ref_subdir, auto_subdir]:
        ndjson_fn = result_dir / subdir / 'measures_ap_whole.ndjson'
        all_data, all_lm_data  = [], []
        for _, row in df_study.iterrows():
            filename = (result_dir / subdir / 'ap_whole' / row['file_id']).with_suffix('.json')
            if filename.stem in all_data_dict[subdir]:
                data = all_data_dict[subdir][filename.stem]
            else:
                with open(filename) as f:
                    data = json.load(f)
            all_data.append(data)
            filename = (result_dir / subdir / 'keypoints/ap_whole' / row['file_id']).with_suffix('.json')
            if filename.stem in all_lm_data_dict[subdir]:
                data = all_lm_data_dict[subdir][filename.stem]
            else:
                with open(filename) as f:
                    data = json.load(f)
            all_lm_data.append(data)
        neut_data = all_data[0]
        all_angles = []
        mt = neut_data['curve_indices']['MT']
        pt = neut_data['curve_indices']['PT']
        lt = neut_data['curve_indices']['TL/L']
        for lm_data, pos in zip(all_lm_data, POS):

            angles = []
            # dwg, points, kps, text_attrs, img = measure_and_draw.init_svg(lm_data, str(image_dir), HEIGHT)
            dwg = measure_and_draw.SpinalDrawing(lm_data, image_dir, HEIGHT)
            points, kps, text_attrs, img = dwg.points, dwg.kps, dwg.text_attrs, dwg.img
            for l in LABEL_NAMES:  # remove C7
                kps[l] = np.delete(kps[l], 0, axis=0)
            sacrum_tltr, vert_corners, disc_corners = measure_utils.get_vert_and_disc_corners(
                kps)
            vert_corners = vert_corners.astype(np.float64)
            sup_cnrs = vert_corners[:, :2]
            inf_cnrs = vert_corners[:, 2:]
            gid = 'MT'
            grp = dwg.add(
                dwg.g(id=gid,
                      stroke='green',
                      stroke_width=measure_and_draw.LINE_WIDTH,
                      fill='none'))
            angle = dwg.draw_cobb(sup_cnrs[mt[0]],
                                    inf_cnrs[mt[1]],
                                    grp,
                                    text_attrs=text_attrs)
            angles.append(angle)
            gid = 'TL/L'
            grp = dwg.add(
                dwg.g(id=gid,
                      stroke='blue',
                      stroke_width=measure_and_draw.LINE_WIDTH,
                      fill='none'))
            angle = dwg.draw_cobb(sup_cnrs[lt[0]],
                                    inf_cnrs[lt[1]],
                                    grp,
                                    text_attrs=text_attrs)
            angles.append(angle)
            gid = 'PT'
            grp = dwg.add(
                dwg.g(id=gid,
                      stroke='red',
                      stroke_width=measure_and_draw.LINE_WIDTH,
                      fill='none'))
            angle = dwg.draw_cobb(sup_cnrs[pt[0]],
                                    inf_cnrs[pt[1]],
                                    grp,
                                    text_attrs=text_attrs)
            angles.append(angle)
            all_angles.append(angles)
            name = f'{pid}_{date}_{subdir}_{pos}.svg'
            # dwg.saveas(out_dir / name)
            svg_names.append(dwg.tostring())
        all_angles = np.array(all_angles)
        left = all_angles[0] - all_angles[1] # neut - left
        right = all_angles[0] - all_angles[2] # neut - right
        angles_per_subdir.append(all_angles)
    angles_per_subdir.append(angles_per_subdir[0]-angles_per_subdir[1]) # true - pred
    rows.append([pid, date] + np.array(angles_per_subdir).ravel().tolist())
    with open(out_image_dir / f'{pid}_{date}.html', 'w') as f:
       
        f.write(styles)
        f.write('<body>')
        f.write('<div class="reference">\n')
        for svg in svg_names[:3]:
            f.write(svg)
            f.write('\n')
        f.write('</div>\n')
        f.write('<div class="AI">\n')
        for svg in svg_names[3:]:
            f.write(svg)
            f.write('\n')
        f.write('</div>\n')
        f.write('</body>')

PREFIX = ['true','pred','t-p']
COLS = ['MT','TL/L','PT']

df = pd.DataFrame(rows, columns=['PatientID','StudyDate']+[' '.join(v) for v in itertools.product(PREFIX, COLS, POS)])
df.dropna(inplace=True)
df.to_csv(out_dir / 'ap_flex.csv', index=False)

In [ ]:
styles = '''
<style>
body {
    display: flex;
    flex-direction: row;
    flex-wrap: wrap;
}
div {
    display: flex;
    flex-direction: column;
}

svg {
    max-height : 1024px;
}
</style>
'''

with open(out_dir / 'catalog.html', 'w') as f:
    f.write(styles)
    f.write('<body>\n')
    for pid, date in pid_date_pairs:
        f.write('<div>')
        link = f'image/{pid}_{date}.html'
        f.write(f'<a href="{link}">{pid}_{date}</a>')
        f.write(f'<iframe width="700" height="1024" src="{link}"></iframe>')
        f.write('</div>')
        # if pid=='OSC00125':
        #     break
    f.write('</body>')
display(HTML(f'<h2>Reference</h2>'))
N_COLS = 9
df_true = df.iloc[:, 2:(N_COLS+2)]
df_true.describe()

In [ ]:
display(HTML(f'<h2>AI</h2>'))
df_pred = df.iloc[:, (N_COLS+2):(2*N_COLS+2)]
df_pred.describe()

In [ ]:
from szkspyne.utils.measure import pearsonr_ci
from scipy import stats
from collections import defaultdict

df_diff = df.iloc[:, (2*N_COLS+2):(3*N_COLS+2)]
desc = df_diff.describe()
new_rows = defaultdict(list)
for col_diff in df_diff.columns:
    col_true = col_diff.replace('t-p','true')
    col_pred = col_diff.replace('t-p','pred')
    X, Y = df[col_true], df[col_pred]

    r, p, lo, hi = pearsonr_ci(X, Y)
    new_rows['r'].append(r)
    new_rows['R^2'].append(r**2)
    diff = X - Y
    new_rows['RMS'].append(np.sqrt((diff**2).mean()))
    result = stats.linregress(X,Y)
    new_rows['regression slope'].append(result.slope)
    new_rows['regression intercept'].append(result.intercept)
for k, v in new_rows.items():
    desc.loc[k] = v  
display(HTML(f'<h2>Reference - AI</h2>'))
desc

In [ ]:
display(HTML(f'<h2>|Reference - AI|</h2>'))
df_diff.abs().describe()